# 09 — Hakaluki Haor inundation dynamics, 2019–2025

**Fast, reproducible screening workflow:** Sentinel-1 C-band GRD → consistent orbit-track selection → monthly low-VV screening and observation-coverage QA → monthly/seasonal figures, 2022 departures → 7 annual frequency rasters.

**Science status: EXPLORATORY — NOT VALIDATED INUNDATION MAPS.** A low radar VV return can indicate smooth open water, but also other smooth surfaces, radar shadow, or acquisition effects; flooded vegetation can return **high** radar backscatter. This workflow reports **low-VV candidate open-water area**, **not total inundated wetland area** or ground-truth flood extent. Its threshold range is **sensitivity analysis, not a statistical confidence interval**. The 2019–2025 results are methodologically different from the earlier 2022 Otsu pilot; compare them only after harmonizing algorithms and validation.

**Design controls:** one `orbitProperties_pass` + `relativeOrbitNumber_start` track throughout; VV/VH+IW metadata filters; incidence-angle mask; 30 m focal median; monthly median compositing; explicit 0-scene months; valid-area denominator; per-pixel annual observed-month count; year-by-year caching tied to parameters. The 30 m grid is an **analysis sampling resolution**, not Sentinel-1's native resolution. Monthly bins use **UTC**, not Bangladesh local time.

**Primary sources:** [Google Earth Engine Sentinel-1 processing](https://developers.google.com/earth-engine/guides/sentinel1); [GEE direct-download limits](https://developers.google.com/earth-engine/apidocs/ee-image-getdownloadurl); [Olofsson et al. (2014), accuracy and area estimation](https://doi.org/10.1016/j.rse.2014.02.015).

**Run all cells in order.** The first server computation can take several minutes. Re-run years from cached local files when the configuration has not changed. No local satellite GeoTIFFs are required beforehand.

**QA revision 09.3 (export-only fix):** Earth Engine direct downloads now specify `crs`, `crs_transform`, and `dimensions` without also specifying `region`, which Earth Engine rejects. Both 09E annual exports and 09G June 2022 export were corrected. Processing config and cached 09C results are unchanged.


In [ ]:
# 09A — configuration and study polygon
from pathlib import Path
from datetime import datetime, timezone
from math import floor, ceil
import hashlib, json, warnings
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import rasterio
from rasterio.transform import from_origin
import requests
import ee

ROOT = Path(r"F:\Wetland Test\Hakaluki_Haor_Wetland_Test_Jupyter_GEE_Project")
BOUNDARY = ROOT / "data/boundary/hakaluki_boundary.gpkg"
if not BOUNDARY.exists():
    BOUNDARY = ROOT / "Study area/Hakaluki_Haor.shp"
assert BOUNDARY.exists(), f"Boundary not found: {BOUNDARY}"

TABLES = ROOT / "results/tables"
FIGURES = ROOT / "results/figures"
RASTERS = ROOT / "results/rasters/inundation_2019_2025"
CACHE = ROOT / "data/processed/inundation09_cache"
for p in (TABLES, FIGURES, RASTERS, CACHE):
    p.mkdir(parents=True, exist_ok=True)

GEE_PROJECT = "axial-paratext-491911-b6"
START_YEAR, END_YEAR = 2019, 2025
TRACK_MODE = "AUTO_BEST_TRACK"  # or "MANUAL"
MANUAL_ORBIT_PASS = "ASCENDING"
MANUAL_RELATIVE_ORBIT = None    # set integer only if TRACK_MODE == "MANUAL"
VV_THRESHOLDS_DB = (-19.0, -17.0, -15.0)  # conservative, nominal, permissive
ANGLE_MIN_DEG, ANGLE_MAX_DEG = 30.0, 45.0
SMOOTH_RADIUS_M = 30              # focal median; screening choice, not validated
ANALYSIS_SCALE_M = 30             # EE reducer sampling
MIN_MONTHLY_COVERAGE = 0.90      # gate for interannual comparisons
MIN_BASELINE_YEARS = 3           # gate for month-matched 2022 departures
DOWNLOAD_ANNUAL_RASTERS = True    # set False to complete charts first
FORCE_RECOMPUTE = False           # True ignores matching year CSV cache
MAX_DOWNLOAD_BYTES = 30 * 1024 * 1024

haor = gpd.read_file(BOUNDARY).to_crs("EPSG:4326")
assert len(haor) and haor.geometry.notna().all() and haor.geometry.is_valid.all()
roi_shape = haor.geometry.union_all()
roi_area_m2 = float(gpd.GeoSeries([roi_shape], crs="EPSG:4326").to_crs("EPSG:32646").area.iloc[0])
assert 1e7 < roi_area_m2 < 5e8, "Unexpected boundary area; check study polygon"
# Earth Engine must be initialized BEFORE constructing ee.Geometry.
try:
    ee.Initialize(project=GEE_PROJECT)
except Exception as exc:
    raise RuntimeError("Earth Engine initialization failed. Authenticate as in Notebook 00.") from exc

ROI = ee.Geometry(roi_shape.__geo_interface__)
# Use the SAME Earth Engine pixel-area reducer grid for area numerators and
# observed-coverage denominator; keep projected polygon area separately.
reference_area_response = (ee.Image.pixelArea().rename("reference_m2")
    .reduceRegion(reducer=ee.Reducer.sum(), geometry=ROI,
        crs="EPSG:32646", scale=ANALYSIS_SCALE_M,
        maxPixels=100_000_000, tileScale=4).getInfo())
reference_area_m2 = float(reference_area_response["reference_m2"])
assert np.isfinite(reference_area_m2) and reference_area_m2 > 0
print("EE sampled ROI area (km²):", round(reference_area_m2 / 1e6, 4))

print("Boundary:", BOUNDARY)
print("Projected ROI area (km²):", round(roi_area_m2/1e6, 3))
print("Period:", START_YEAR, "through", END_YEAR)
print("Analysis:", ANALYSIS_SCALE_M, "m; thresholds:", VV_THRESHOLDS_DB)
print("Area and classification remain preliminary.")

## 09B — Discover consistent Sentinel-1 observation tracks

Do **not** silently mix ascending and descending passes or different relative orbits. We first inspect *all* IW dual-VV/VH scenes and select the track covering the most distinct calendar months (tie-break: scene count). If coverage is poor, record that before interpreting missing months. The selected track is **not necessarily optimal for every subarea**, so the spatial observation mask remains essential.

In [ ]:
# 09B — catalogue inventory and track selection (small metadata transfer only)
BASE = (ee.ImageCollection("COPERNICUS/S1_GRD")
    .filterBounds(ROI)
    .filterDate(f"{START_YEAR}-01-01", f"{END_YEAR+1}-01-01")
    .filter(ee.Filter.eq("instrumentMode", "IW"))
    .filter(ee.Filter.listContains("transmitterReceiverPolarisation", "VV"))
    .filter(ee.Filter.listContains("transmitterReceiverPolarisation", "VH"))
)
scene_total = int(BASE.size().getInfo())
if scene_total == 0:
    raise RuntimeError("No dual VV/VH Sentinel-1 IW scenes found; check ROI and dates.")
if scene_total > 5000:
    raise RuntimeError(f"Unexpected scene count ({scene_total}); narrow the study area.")

# Aggregations are index-aligned for the same sorted ImageCollection.
BASE = BASE.sort("system:time_start")
times = BASE.aggregate_array("system:time_start").getInfo()
passes = BASE.aggregate_array("orbitProperties_pass").getInfo()
orbits = BASE.aggregate_array("relativeOrbitNumber_start").getInfo()
scene_ids = BASE.aggregate_array("system:index").getInfo()
assert all(len(v) == scene_total for v in (times, passes, orbits, scene_ids))

scene_df = pd.DataFrame({
    "scene_id": scene_ids,
    "acquisition_utc": pd.to_datetime(times, unit="ms", utc=True),
    "pass": passes,
    "relative_orbit": orbits,
})
scene_df["relative_orbit"] = pd.to_numeric(scene_df["relative_orbit"], errors="raise").astype(int)
scene_df["year_month"] = scene_df["acquisition_utc"].dt.strftime("%Y-%m")
scene_df["year"] = scene_df["acquisition_utc"].dt.year
scene_df.to_csv(TABLES / "inundation09_s1_scene_inventory.csv", index=False)

track_df = (scene_df.groupby(["pass", "relative_orbit"], as_index=False)
  .agg(scenes=("scene_id", "size"), distinct_months=("year_month", "nunique"),
       earliest=("acquisition_utc", "min"), latest=("acquisition_utc", "max"))
  .sort_values(["distinct_months", "scenes"], ascending=[False, False])
  .reset_index(drop=True))
track_df["monthly_coverage_percent"] = 100*track_df["distinct_months"] / (12*(END_YEAR-START_YEAR+1))
track_df.to_csv(TABLES / "inundation09_track_selection.csv", index=False)
print("Candidate scenes:", scene_total)
print("Track candidates:")
display(track_df)

if TRACK_MODE == "AUTO_BEST_TRACK":
    chosen_pass = str(track_df.iloc[0]["pass"])
    chosen_orbit = int(track_df.iloc[0]["relative_orbit"])
elif TRACK_MODE == "MANUAL":
    if MANUAL_RELATIVE_ORBIT is None:
        raise ValueError("Set MANUAL_RELATIVE_ORBIT when TRACK_MODE='MANUAL'.")
    chosen_pass, chosen_orbit = MANUAL_ORBIT_PASS, int(MANUAL_RELATIVE_ORBIT)
    if not ((track_df["pass"] == chosen_pass) & (track_df["relative_orbit"] == chosen_orbit)).any():
        raise RuntimeError("Requested manual track has no matching scenes.")
else:
    raise ValueError("TRACK_MODE must be AUTO_BEST_TRACK or MANUAL")

selected_scenes = scene_df[(scene_df["pass"] == chosen_pass) & (scene_df["relative_orbit"] == chosen_orbit)].copy()
selected_counts = selected_scenes.groupby("year_month").size().to_dict()
print("\nSELECTED TRACK:", chosen_pass, "relative orbit", chosen_orbit)
print("Distinct months:", len(selected_counts), "of", 12*(END_YEAR-START_YEAR+1))
print("Selected scenes:", len(selected_scenes))
if len(selected_counts) < 0.65*12*(END_YEAR-START_YEAR+1):
    warnings.warn("Selected single track observes fewer than 65% of months. Seasonal comparisons may be sparse. Do not fill missing months with zeros.")

TRACK = (BASE.filter(ee.Filter.eq("orbitProperties_pass", chosen_pass))
             .filter(ee.Filter.eq("relativeOrbitNumber_start", chosen_orbit)))

CONFIG = dict(version="09.2-preserved-mask-matched-area", years=[START_YEAR, END_YEAR],
    pass_name=chosen_pass, relative_orbit=chosen_orbit,
    vv_thresholds_db=list(VV_THRESHOLDS_DB), angle_limits=[ANGLE_MIN_DEG, ANGLE_MAX_DEG],
    focal_median_radius_m=SMOOTH_RADIUS_M, preserve_original_scene_mask=True,
    reduce_scale_m=ANALYSIS_SCALE_M, pixel_area_denominator="EE_same_grid",
    min_coverage=MIN_MONTHLY_COVERAGE,
    roi_wkb_sha256=hashlib.sha256(roi_shape.wkb).hexdigest(),
    ee_collection="COPERNICUS/S1_GRD", monthly_time_zone="UTC")
CONFIG_HASH = hashlib.sha256(json.dumps(CONFIG, sort_keys=True).encode()).hexdigest()[:16]
(CONFIG | {"config_hash": CONFIG_HASH, "status": "exploratory_not_validated"})
(TABLES / "inundation09_processing_parameters.json").write_text(
    json.dumps(CONFIG | {"config_hash": CONFIG_HASH, "status": "exploratory_not_validated"}, indent=2), encoding="utf-8")
print("Configuration fingerprint:", CONFIG_HASH)

**QA revision 09.2.** This version restores each original Sentinel-1 scene's valid-data mask after focal-median filtering and calculates coverage using the *same* Earth Engine pixel-area reduction grid as the numerator. It uses a different configuration hash, so **09C recomputes the seven annual CSV checkpoints**; do not mix 09.1 and 09.2 products. Maps are low-VV screening frequency, not validated inundation probabilities.

## 09C — Monthly low-VV screening, coverage and sensitivity (84 months)

Each month uses a **median** of consistent-track VV observations, after an incidence-angle mask and a 30 m median speckle-suppression operation. Zero-scene months stay missing (`NaN`), never zero water. Areas are computed on a **30 m UTM sampling grid**; they are pixel-based **screening areas**, not accuracy-adjusted estimates.

The three fixed VV thresholds (`−19`, `−17`, `−15` dB) are transparent *illustrative* sensitivity choices. **Do not label them an optimal threshold or 95% uncertainty interval.** A pass/relative-orbit control alone does not eliminate incidence-angle drift, topographic effects, or satellite/platform differences.

In [ ]:
# 09C — EE monthly workflow with annual checkpoint caches
from calendar import monthrange

def get_month_vv(year: int, month: int):
    start = f"{year}-{month:02d}-01"
    end = f"{year+1}-01-01" if month == 12 else f"{year}-{month+1:02d}-01"
    images = TRACK.filterDate(start, end)
    def prep(image):
        angle = image.select("angle")
        vv = image.select("VV")
        # Mask invalid viewing angles before spatial filtering.
        vv = vv.updateMask(angle.gte(ANGLE_MIN_DEG).And(angle.lte(ANGLE_MAX_DEG)))
        # Spatial smoothing can expand support at image-mask boundaries.
        # Restore each scene's ORIGINAL angle-filtered valid-data support.
        smoothed = vv.focal_median(radius=SMOOTH_RADIUS_M,
                                   kernelType="circle", units="meters")
        return smoothed.updateMask(vv.mask()).rename("VV")
    return images.map(prep).median().rename("VV")

def monthly_ee_feature(year: int, month: int):
    vv = get_month_vv(year, month)
    # Masked observation pixels are explicitly zero for the AREA SUM only.
    observed = vv.mask().unmask(0).gt(0)
    px = ee.Image.pixelArea()
    bands = [px.multiply(observed).rename("observed_m2")]
    band_names = ("area_m19_m2", "area_m17_m2", "area_m15_m2")
    for threshold, band_name in zip(VV_THRESHOLDS_DB, band_names):
        is_low_vv = vv.lte(float(threshold)).unmask(0).And(observed)
        bands.append(px.multiply(is_low_vv).rename(band_name))
    area_values = ee.Image.cat(bands).reduceRegion(
        reducer=ee.Reducer.sum(), geometry=ROI,
        crs="EPSG:32646", scale=ANALYSIS_SCALE_M,
        maxPixels=100_000_000, tileScale=4)
    return ee.Feature(None, area_values).set({"year": year, "month": month})

records = []
for year in range(START_YEAR, END_YEAR+1):
    cache_path = CACHE / f"year_{year}_{CONFIG_HASH}.csv"
    if cache_path.exists() and not FORCE_RECOMPUTE:
        cached = pd.read_csv(cache_path)
        if len(cached) == 12 and sorted(cached["month"].astype(int).tolist()) == list(range(1,13)):
            records.extend(cached.to_dict("records"))
            print(f"{year}: loaded matching cached results (12 months)")
            continue

    missing_records, active_features = [], []
    for month in range(1, 13):
        ym = f"{year}-{month:02d}"
        n_scenes = int(selected_counts.get(ym, 0))
        if n_scenes == 0:
            missing_records.append(dict(year=year, month=month, scene_count=0,
                observed_m2=np.nan, area_m19_m2=np.nan,
                area_m17_m2=np.nan, area_m15_m2=np.nan))
        else:
            active_features.append(monthly_ee_feature(year, month))
    print(f"{year}: requesting {len(active_features)} observed months from Earth Engine...")
    year_records = list(missing_records)
    if active_features:
        result = ee.FeatureCollection(active_features).getInfo()
        for feature in result["features"]:
            prop = feature["properties"]
            ym = f"{int(prop['year'])}-{int(prop['month']):02d}"
            year_records.append({
                "year": int(prop["year"]), "month": int(prop["month"]),
                "scene_count": int(selected_counts.get(ym, 0)),
                "observed_m2": prop.get("observed_m2"),
                "area_m19_m2": prop.get("area_m19_m2"),
                "area_m17_m2": prop.get("area_m17_m2"),
                "area_m15_m2": prop.get("area_m15_m2")})
    year_frame = pd.DataFrame(year_records).sort_values("month")
    assert len(year_frame) == 12 and year_frame["month"].nunique() == 12
    year_frame["config_hash"] = CONFIG_HASH
    year_frame.to_csv(cache_path, index=False)
    records.extend(year_frame.to_dict("records"))
    print(f"{year}: checkpoint saved")

monthly = pd.DataFrame(records).sort_values(["year", "month"]).reset_index(drop=True)
assert len(monthly) == 84 and monthly[["year","month"]].drop_duplicates().shape[0] == 84
monthly["date"] = pd.to_datetime(dict(year=monthly.year, month=monthly.month, day=1))
for tag in ("m19", "m17", "m15"):
    monthly[f"lowvv_{tag}_km2"] = pd.to_numeric(monthly[f"area_{tag}_m2"], errors="coerce")/1e6
monthly["observed_km2"] = pd.to_numeric(monthly["observed_m2"], errors="coerce")/1e6
monthly["observed_fraction"] = monthly["observed_m2"] / reference_area_m2
# Never hide a material coverage inconsistency by indiscriminate clipping.
finite_coverage = monthly.loc[monthly.scene_count > 0, "observed_fraction"].dropna()
if finite_coverage.empty:
    raise RuntimeError("No months with valid coverage estimates")
if not finite_coverage.between(0, 1.001).all():
    raise RuntimeError("Coverage outside [0,1.001]: audit EE observation masks and reducer")
monthly["observed_fraction"] = monthly["observed_fraction"].clip(lower=0, upper=1.0)
monthly["ee_reference_roi_km2"] = reference_area_m2 / 1e6
monthly["coverage_definition"] = "original_S1_valid_mask_after_angle_filter;EE_pixelArea_same_grid"
monthly.loc[monthly.scene_count == 0, "observed_fraction"] = 0
monthly["comparison_eligible"] = ((monthly.scene_count >= 1) &
                                    (monthly.observed_fraction >= MIN_MONTHLY_COVERAGE))
monthly["single_scene_warning"] = monthly.scene_count == 1
monthly["nominal_lowvv_km2"] = monthly["lowvv_m17_km2"]
monthly["lowvv_fraction_observed"] = monthly.nominal_lowvv_km2/monthly.observed_km2
if (monthly.loc[monthly.scene_count > 0, "lowvv_fraction_observed"] > 1.001).any():
    raise RuntimeError("Low-VV area exceeds observed area; inspect masks/reducers")
monthly["sensitivity_span_km2"] = monthly.lowvv_m15_km2 - monthly.lowvv_m19_km2
monthly["config_hash"] = CONFIG_HASH

has_all_areas = monthly[["lowvv_m19_km2", "lowvv_m17_km2", "lowvv_m15_km2"]].notna().all(axis=1)
monotonic = ((monthly.lowvv_m19_km2 <= monthly.lowvv_m17_km2 + 0.05) &
             (monthly.lowvv_m17_km2 <= monthly.lowvv_m15_km2 + 0.05))
if not monotonic[has_all_areas].all():
    warnings.warn("Unexpected non-monotonic threshold areas. Check EE reductions.")
monthly.to_csv(TABLES / "inundation09_monthly_2019_2025.csv", index=False)
print("\n=== 84-MONTH COVERAGE ===")
print("Months without scenes:", int((monthly.scene_count==0).sum()))
print("Months meeting coverage criterion:", int(monthly.comparison_eligible.sum()))
print("Months relying on a single scene:", int(monthly.single_scene_warning.sum()))
display(monthly[["date","scene_count","observed_fraction","comparison_eligible",
    "lowvv_m19_km2","nominal_lowvv_km2","lowvv_m15_km2"]].head(24))

## 09D — Coverage-aware seasonal statistics and 2022 departures

Only months with at least the required **90% observed ROI** enter area comparisons. The 2022 month-matched reference is the median among eligible observations from **2019–2021 and 2023–2025**; require at least 3 such years. These are **descriptive departures**, not significance tests or causal flood attribution. Missing months are blank in the heatmap.

In [ ]:
# 09D — statistical tables and figures (local; fast)
eligible = monthly[monthly.comparison_eligible].copy()
annual = (monthly.groupby("year", as_index=False)
    .agg(months_with_scene=("scene_count", lambda x: int((x>0).sum())),
         eligible_months=("comparison_eligible","sum"),
         single_scene_months=("single_scene_warning","sum"),
         median_observed_fraction=("observed_fraction","median")))
annual["median_nominal_lowvv_km2"] = annual["year"].map(
    eligible.groupby("year")["nominal_lowvv_km2"].median())
annual.to_csv(TABLES / "inundation09_annual_coverage_qa.csv", index=False)

baseline = (eligible[eligible.year != 2022].groupby("month")
    .agg(baseline_median_km2=("nominal_lowvv_km2", "median"),
         n_baseline_years=("year", "nunique")).reset_index())
compare_2022 = monthly[monthly.year == 2022][["month","date","comparison_eligible",
    "scene_count","observed_fraction","nominal_lowvv_km2"]].merge(baseline, on="month", how="left")
compare_2022["baseline_usable"] = compare_2022.n_baseline_years.ge(MIN_BASELINE_YEARS)
compare_2022["departure_km2"] = (compare_2022.nominal_lowvv_km2 - compare_2022.baseline_median_km2)
compare_2022.loc[~(compare_2022.comparison_eligible & compare_2022.baseline_usable), "departure_km2"] = np.nan
compare_2022.to_csv(TABLES / "inundation09_2022_month_matched_departures.csv", index=False)

print("=== ANNUAL OBSERVATION QA ===")
display(annual)
print("\n=== 2022 MONTH-MATCHED DEPARTURES (ELIGIBLE ONLY) ===")
display(compare_2022)

fig, axes = plt.subplots(2, 1, figsize=(13, 8), sharex=True, constrained_layout=True)
axes[0].plot(monthly.date, monthly.nominal_lowvv_km2.where(monthly.comparison_eligible),
             marker="o", markersize=3, linewidth=1.2, label="Nominal VV ≤ −17 dB")
axes[0].fill_between(monthly.date,
    monthly.lowvv_m19_km2.where(monthly.comparison_eligible).to_numpy(dtype=float),
    monthly.lowvv_m15_km2.where(monthly.comparison_eligible).to_numpy(dtype=float),
    alpha=0.20, label="Threshold sensitivity (−19 to −15 dB)")
axes[0].set_ylabel("Low-VV screening area (km²)")
axes[0].legend(loc="upper left")
axes[0].grid(alpha=0.2)
axes[0].set_title("2019–2025 | Comparable months only (≥90% sampled ROI coverage)")
axes[1].bar(monthly.date, monthly.observed_fraction.clip(upper=1), width=23)
axes[1].axhline(MIN_MONTHLY_COVERAGE, linestyle="--", color="black", linewidth=1)
axes[1].set_ylabel("ROI observed fraction")
axes[1].set_ylim(0, 1.08)
axes[1].grid(alpha=0.2)
axes[1].set_xlabel("Month (UTC)")
fig.suptitle("Hakaluki Haor: preliminary Sentinel-1 low-VV dynamics and availability")
fig.savefig(FIGURES/"inundation09_monthly_screening_timeseries.png", dpi=230, bbox_inches="tight")
plt.show(); plt.close(fig)

heat = monthly.pivot(index="year", columns="month", values="nominal_lowvv_km2")
heat_elig = monthly.pivot(index="year", columns="month", values="comparison_eligible")
heat = heat.where(heat_elig)
fig, ax = plt.subplots(figsize=(12, 4.6), constrained_layout=True)
im = ax.imshow(np.ma.masked_invalid(heat.to_numpy(dtype=float)), aspect="auto", cmap="viridis")
ax.set_yticks(range(len(heat.index)), labels=heat.index)
ax.set_xticks(range(12), labels=["Jan","Feb","Mar","Apr","May","Jun","Jul","Aug","Sep","Oct","Nov","Dec"])
ax.set_xlabel("Calendar month (UTC)"); ax.set_ylabel("Year")
ax.set_title("Low-VV area by month (blank = insufficient observation coverage)")
fig.colorbar(im, ax=ax, label="Nominal low-VV area (km²)")
fig.savefig(FIGURES/"inundation09_month_year_heatmap.png", dpi=230, bbox_inches="tight")
plt.show(); plt.close(fig)

fig, ax = plt.subplots(figsize=(11, 4.8), constrained_layout=True)
xx = compare_2022.month.to_numpy()
ax.axhline(0, color="black", linewidth=1)
ax.bar(xx, compare_2022.departure_km2)
ax.set_xticks(range(1,13), labels=["Jan","Feb","Mar","Apr","May","Jun","Jul","Aug","Sep","Oct","Nov","Dec"])
ax.set_ylabel("2022 minus reference median (km²)")
ax.set_title("2022 low-VV monthly departures | reference: other eligible years (n ≥ 3)")
ax.grid(axis="y", alpha=0.2)
fig.savefig(FIGURES/"inundation09_2022_month_matched_departures.png", dpi=230, bbox_inches="tight")
plt.show(); plt.close(fig)

print("Saved monthly, coverage and 2022 comparison figures. Do not interpret differences as verified flood area.")

## 09E — Download annual per-pixel frequency maps (7 GeoTIFFs)

Each valid month contributes **one** binary low-VV state derived from its median. `observed_months` counts months with a valid radar observation at each pixel. `lowvv_months` counts detected months. `lowvv_frequency = lowvv_months / observed_months` where denominator > 0. Missing months are **not** treated as dry, and frequency is **not a probability of flooding**. Track availability differences across years can bias comparisons; always display the observed-month count.

Direct Earth Engine downloads are capped (32 MB/request), so rasters are limited to the study-area 30 m target grid. Seven annual GeoTIFFs should usually fit, but computation or download may take several minutes. Set `DOWNLOAD_ANNUAL_RASTERS=False` above to defer only this stage.

In [ ]:
# 09E — annual frequencies: request and locally verify 7 small GeoTIFFs
if not DOWNLOAD_ANNUAL_RASTERS:
    print("Annual downloads disabled. Set DOWNLOAD_ANNUAL_RASTERS=True in 09A and rerun 09E.")
else:
    import math
    roi_utm = gpd.GeoSeries([roi_shape], crs="EPSG:4326").to_crs("EPSG:32646").iloc[0]
    xmin, ymin, xmax, ymax = roi_utm.bounds
    GRID = 30.0
    left = floor(xmin/GRID)*GRID; right = ceil(xmax/GRID)*GRID
    bottom = floor(ymin/GRID)*GRID; top = ceil(ymax/GRID)*GRID
    WIDTH = int(round((right-left)/GRID))
    HEIGHT = int(round((top-bottom)/GRID))
    TRANSFORM = from_origin(left, top, GRID, GRID)
    assert WIDTH*HEIGHT < 2_000_000
    print("Annual export grid:", WIDTH, "x", HEIGHT, "EPSG:32646", GRID, "m")

    def annual_ee_image(year):
        valid_images, lowvv_images = [], []
        for month in range(1, 13):
            if not selected_counts.get(f"{year}-{month:02d}", 0):
                continue
            vv = get_month_vv(year, month)
            valid = vv.mask().unmask(0).gt(0).rename("valid")
            low = vv.lte(VV_THRESHOLDS_DB[1]).unmask(0).And(valid).rename("lowvv")
            valid_images.append(valid.toFloat())
            lowvv_images.append(low.toFloat())
        if not valid_images:
            # No track scenes: return a fully invalid frequency and zero monthly counts.
            zeros = ee.Image.constant(0).rename("observed_months").toFloat()
            freq = ee.Image.constant(0).updateMask(ee.Image.constant(0)).rename("lowvv_frequency")
            wet = ee.Image.constant(0).rename("lowvv_months").toFloat()
        else:
            valid_count = ee.ImageCollection.fromImages(valid_images).sum().toFloat()
            wet = ee.ImageCollection.fromImages(lowvv_images).sum().rename("lowvv_months").toFloat()
            zeros = valid_count.rename("observed_months")
            freq = wet.divide(zeros).updateMask(zeros.gt(0)).rename("lowvv_frequency")
        return ee.Image.cat([freq, zeros, wet]).toFloat().clip(ROI).unmask(-9999, sameFootprint=False)

    annual_files = []
    for year in range(START_YEAR, END_YEAR+1):
        dest = RASTERS / f"HAKALUKI_{year}_{CONFIG_HASH}_lowVV_frequency_30m.tif"
        if not dest.exists() or FORCE_RECOMPUTE:
            print(f"{year}: preparing EE annual frequency raster...")
            image = annual_ee_image(year)
            url = image.getDownloadURL(dict(format="GEO_TIFF", crs="EPSG:32646",
                                            crs_transform=list(TRANSFORM)[:6],
                                            dimensions=[WIDTH,HEIGHT]))
            temp = dest.with_suffix(".part")
            size = 0
            try:
                with requests.get(url, stream=True, timeout=(30, 240)) as r:
                    r.raise_for_status()
                    with temp.open("wb") as f:
                        for chunk in r.iter_content(1024*1024):
                            if not chunk: continue
                            size += len(chunk)
                            if size > MAX_DOWNLOAD_BYTES:
                                raise RuntimeError(f"{year}: download exceeded local safety limit")
                            f.write(chunk)
                with rasterio.open(temp) as check:
                    assert check.count == 3 and check.width == WIDTH and check.height == HEIGHT
                    assert check.crs is not None and check.crs.to_epsg() == 32646
                    assert check.transform.almost_equals(TRANSFORM), "Export grid differs"
                temp.replace(dest)
            finally:
                if temp.exists(): temp.unlink()
        with rasterio.open(dest) as check:
            assert check.count == 3 and check.crs.to_epsg()==32646
            assert check.transform.almost_equals(TRANSFORM)
            assert check.width == WIDTH and check.height == HEIGHT
            # Internal consistency; frequency undefined where no months observed.
            sample = check.read(masked=True)
            frequency, observed_count, low_count = [
                np.asarray(a.astype(np.float64).filled(np.nan)) for a in sample
            ]
            has_obs = (np.isfinite(observed_count) & (observed_count > 0))
            if has_obs.any():
                assert np.all((observed_count[has_obs] <= 12.001) &
                              (observed_count[has_obs] >= 1))
                assert np.all((frequency[has_obs] >= -1e-6) &
                              (frequency[has_obs] <= 1 + 1e-6))
                assert np.all((low_count[has_obs] >= -1e-6) &
                              (low_count[has_obs] <= observed_count[has_obs] + 1e-5))
        annual_files.append(dest)
        print(year, "verified:", dest.name, round(dest.stat().st_size/1e6,2), "MB")
    print("All", len(annual_files), "annual raster files verified.")

## 09F — Annual spatial map gallery and observations (local)

A year with only a few valid months cannot be interpreted as a robust *annual* flooding climatology. The `observed_months` denominator map is an indispensable QA panel. This figure suppresses display where fewer than **three** valid months were observed per pixel; adjust after examining actual data availability, not to force a desired result.

In [ ]:
# 09F — 7 annual maps + observation-count map. Requires 09E downloads.
from matplotlib.colors import Normalize
from rasterio.features import geometry_mask
from shapely.geometry import mapping
files = [RASTERS / f"HAKALUKI_{year}_{CONFIG_HASH}_lowVV_frequency_30m.tif" for year in range(START_YEAR, END_YEAR+1)]
if not all(f.exists() for f in files):
    print("Annual raster gallery skipped: run 09E to download the missing annual GeoTIFFs.")
else:
    haor_utm = haor.to_crs("EPSG:32646")
    fig, axes = plt.subplots(2,4, figsize=(17,9), constrained_layout=True)
    im = None
    raster_qa = []
    for ax, year, file in zip(axes.flat[:7], range(START_YEAR,END_YEAR+1), files):
        with rasterio.open(file) as src:
            freq = src.read(1, masked=True).astype(float).filled(np.nan)
            count = src.read(2, masked=True).astype(float).filled(np.nan)
            extent = [src.bounds.left,src.bounds.right,src.bounds.bottom,src.bounds.top]
            roi_mask = geometry_mask([mapping(haor_utm.geometry.union_all())],
                                     out_shape=src.shape, transform=src.transform, invert=True)
            sufficient = roi_mask & (count >= 3) & np.isfinite(freq)
            visible = np.where(sufficient, freq, np.nan)
            roi_vals = count[roi_mask & np.isfinite(count) & (count >= 0)]
            raster_qa.append(dict(year=year, raster=file.name,
                count_median=float(np.median(roi_vals)) if len(roi_vals) else np.nan,
                percent_roi_pixels_with_3plus_months=100*float(np.mean(sufficient[roi_mask]))))
            im = ax.imshow(visible, extent=extent, origin="upper", vmin=0, vmax=1, cmap="viridis")
            haor_utm.boundary.plot(ax=ax, color="black", linewidth=.65)
            ax.set_title(f"{year} | months observed (median): {np.median(roi_vals):.0f}" if len(roi_vals) else str(year))
            ax.ticklabel_format(style="plain", axis="both")
            ax.set_aspect("equal")
    axes.flat[7].axis("off")
    
if im is not None:
    fig.colorbar(
        im,
        ax=list(axes.flat),
        label="Low-VV detection frequency",
        shrink=0.7
    )

    fig.suptitle("2019–2025 | preliminary low-VV occurrence on one Sentinel-1 track\nDisplay only where ≥3 valid months/year; not validated inundation probability", fontsize=14)
    out = FIGURES / "inundation09_annual_lowvv_frequency_2019_2025.png"
    fig.savefig(out, dpi=240, bbox_inches="tight")
    plt.show(); plt.close(fig)
    pd.DataFrame(raster_qa).to_csv(TABLES / "inundation09_annual_raster_coverage_qa.csv", index=False)
    display(pd.DataFrame(raster_qa))
    print("Saved annual map:", out)

## 09G — Interpretation and essential validation gates

1. **Read the track-selection table and monthly coverage table before claiming trends.** Missing months and poor spatial coverage cannot be equated with zero inundation; the single-track method may leave a substantial fraction of the record unobserved.
2. **Compare 2022 against other years only within eligible, month-matched observations.** The threshold area alone is not evidence of flood *causation* or exceptional hydrological disturbance.
3. **No independent validation has been performed.** Required next work: stratified probability sample of reference locations, reference imagery acquired close to each SAR date, separate open-water / exposed / flooded-vegetation classes, analyst blinding where possible, error matrix, and uncertainty-adjusted area estimates ([Olofsson et al. 2014](https://doi.org/10.1016/j.rse.2014.02.015)).
4. **Method sensitivity:** test alternative VV thresholds, orbit tracks, filtering scales, and per-scene versus per-month composites. Coherent wetland dynamics require distinguishing radar geometry and submerged/emergent vegetation from dark open-water screening.
5. **Hydroclimatic drivers and methane:** add CHIRPS rainfall, ERA5-Land temperature, and TROPOMI XCH₄ only after aligning time bins and retrieval-quality controls. The XCH₄ column is **not** wetland methane flux.
6. **Area provenance:** the projected polygon area printed in 09A is the denominator for this notebook. Prior EE spherical/geodesic area estimates may differ; do not silently conflate them.

**Suggested next project milestone:** independent SAR-water validation + matched hydroclimatic predictors, followed by a sensitivity analysis covering additional Sentinel-1 tracks. Only validated estimates should replace the current *preliminary screening* figures on the repository homepage.

### 09G — June 2022 spatial departure map

This comparison maps a binary June 2022 low-VV screening indicator minus the June reference frequency from the other six years. It reports valid reference-year count per pixel, masks locations with fewer than three comparable baseline observations, and **does not** classify flooding or estimate statistical significance.

In [ ]:
# 09G — June 2022 spatial low-VV departure relative to six other Junes
# Not a validated flood extent or statistical significance map.
from shapely.geometry import mapping
from rasterio.features import geometry_mask

if not DOWNLOAD_ANNUAL_RASTERS:
    print("Spatial June 2022 anomaly skipped: enable DOWNLOAD_ANNUAL_RASTERS in 09A")
else:
    # Reuse the 30 m grid defined by 09E, which must have run first.
    assert "TRANSFORM" in globals() and "WIDTH" in globals() and "HEIGHT" in globals(), "Run 09E first"
    years_baseline = [y for y in range(START_YEAR, END_YEAR+1) if y != 2022]
    assert len(years_baseline) == 6
    threshold = float(VV_THRESHOLDS_DB[1])

    def june_components(year):
        vv = get_month_vv(year, 6)
        observed = vv.mask().unmask(0).gt(0)
        low = vv.lte(threshold).unmask(0).And(observed)
        return observed.toFloat(), low.toFloat()

    observed_2022, low_2022 = june_components(2022)
    baseline_valid, baseline_low = [], []
    for year in years_baseline:
        obs, low = june_components(year)
        baseline_valid.append(obs)
        baseline_low.append(low)

    count_baseline = ee.ImageCollection.fromImages(baseline_valid).sum().rename("baseline_n_years")
    freq_baseline = (ee.ImageCollection.fromImages(baseline_low).sum()
                     .divide(count_baseline).rename("baseline_june_frequency"))
    valid = observed_2022.gt(0).And(count_baseline.gte(MIN_BASELINE_YEARS))
    june_image = ee.Image.cat([
        low_2022.rename("june2022_lowvv"),
        freq_baseline.rename("baseline_june_frequency"),
        low_2022.subtract(freq_baseline).rename("june2022_minus_baseline"),
        count_baseline.rename("baseline_n_years")
    ]).updateMask(valid).toFloat().clip(ROI).unmask(-9999, sameFootprint=False)

    export = RASTERS / f"HAKALUKI_JUNE2022_vs_2019_2025_{CONFIG_HASH}_30m.tif"
    if not export.exists() or FORCE_RECOMPUTE:
        print("Downloading June 2022 spatial screening departure...")
        signed_url = june_image.getDownloadURL({
            "format": "GEO_TIFF", "crs": "EPSG:32646",
            "crs_transform": list(TRANSFORM)[:6],
            "dimensions": [WIDTH, HEIGHT]
        })
        part = export.with_suffix(".part")
        try:
            with requests.get(signed_url, stream=True, timeout=(30,240)) as resp:
                resp.raise_for_status()
                total = 0
                with part.open("wb") as f:
                    for chunk in resp.iter_content(1024*1024):
                        if chunk:
                            total += len(chunk)
                            if total > MAX_DOWNLOAD_BYTES:
                                raise RuntimeError("June 2022 download exceeds safety limit")
                            f.write(chunk)
            with rasterio.open(part) as src:
                assert src.count == 4 and src.crs.to_epsg()==32646
                assert src.shape == (HEIGHT, WIDTH) and src.transform.almost_equals(TRANSFORM)
            part.replace(export)
        finally:
            if part.exists(): part.unlink()

    with rasterio.open(export) as src:
        assert src.count == 4 and src.crs.to_epsg() == 32646
        assert src.shape == (HEIGHT, WIDTH) and src.transform.almost_equals(TRANSFORM)
        june_binary, baseline_freq, anomaly, baseline_count = [
            src.read(k, masked=True).astype(np.float64).filled(np.nan) for k in range(1,5)
        ]
        extent = [src.bounds.left,src.bounds.right,src.bounds.bottom,src.bounds.top]
        inside = geometry_mask([mapping(haor.to_crs("EPSG:32646").geometry.union_all())],
                               out_shape=src.shape, transform=src.transform, invert=True)
        valid = inside & np.isfinite(anomaly) & np.isfinite(baseline_count) & (baseline_count >= MIN_BASELINE_YEARS)
        assert valid.any(), "No June 2022 pixels with sufficient reference observations"
        assert np.nanmin(anomaly[valid]) >= -1.00001 and np.nanmax(anomaly[valid]) <= 1.00001
        assert np.nanmin(baseline_freq[valid]) >= -1e-6 and np.nanmax(baseline_freq[valid]) <= 1.00001
        assert np.nanmin(june_binary[valid]) >= -1e-6 and np.nanmax(june_binary[valid]) <= 1.00001

    print("June 2022 spatial QA")
    print("ROI pixels on 30m grid:", int(inside.sum()))
    print("Pixels with 2022 observation and >=3 baseline June years:", int(valid.sum()))
    print("Percentage with sufficient June comparisons:", round(100*valid.sum()/inside.sum(),2))
    print("Median 2022-minus-baseline screening departure:", round(float(np.nanmedian(anomaly[valid])), 4))
    print("CAUTION: A per-pixel binary-minus-frequency anomaly is not an area-change map or a p-value")

    labels = ["June 2022 low-VV detection (0/1)",
              "Other-year June low-VV frequency (2019–2025 excluding 2022)",
              "June 2022 detection minus baseline frequency"]
    layers = [june_binary, baseline_freq, anomaly]
    limits = [(0,1),(0,1),(-1,1)]
    cmaps = ["viridis","viridis","RdBu_r"]
    fig, axes = plt.subplots(1,3,figsize=(17,6), constrained_layout=True)
    boundary_utm = haor.to_crs("EPSG:32646")
    for ax, data, label, (lo,hi), cmap in zip(axes,layers,labels,limits,cmaps):
        im = ax.imshow(np.where(valid,data,np.nan),extent=extent,origin="upper",
                       vmin=lo,vmax=hi,cmap=cmap,interpolation="nearest")
        boundary_utm.boundary.plot(ax=ax,color="black",linewidth=.65)
        ax.set_title(label, fontsize=10)
        ax.set_xlabel("UTM 46N Easting (m)")
        ax.set_ylabel("UTM 46N Northing (m)")
        ax.set_aspect("equal")
        fig.colorbar(im, ax=ax, fraction=.045, pad=.02)
    fig.suptitle("Hakaluki | June 2022 low-VV spatial screening departure\n"
                 "Single-track Sentinel-1; unvalidated water indicator",fontsize=13)
    figpath = FIGURES/"inundation09_june2022_spatial_lowvv_departure.png"
    fig.savefig(figpath,dpi=240,bbox_inches="tight")
    plt.show(); plt.close(fig)
    pd.DataFrame([{
        "config_hash": CONFIG_HASH, "target_month": "2022-06",
        "comparison_years": ",".join(str(y) for y in years_baseline),
        "min_reference_years": MIN_BASELINE_YEARS,
        "roi_grid_pixels": int(inside.sum()), "eligible_grid_pixels": int(valid.sum()),
        "eligible_fraction": float(valid.sum()/inside.sum()),
        "median_binary_minus_frequency": float(np.nanmedian(anomaly[valid])),
        "is_validated_flood_extent": False,
        "source_raster": str(export.relative_to(ROOT))
    }]).to_csv(TABLES/"inundation09_june2022_spatial_qa.csv",index=False)
    print("Saved raster:",export)
    print("Saved figure:",figpath)


In [ ]:
# 09H — concise hand-off manifest for the GitHub research log (no automatic public claim)
manifest = {
    "notebook": "09_inundation_dynamics_2019_2025_QA_v3_export_fixed.ipynb",
    "period": f"{START_YEAR}-{END_YEAR}",
    "method_status": "exploratory_lowVV_screening_not_validated_flood_map",
    "reference": "COPERNICUS/S1_GRD; consistent IW VV/VH track",
    "pass": chosen_pass, "relative_orbit": chosen_orbit,
    "config_hash": CONFIG_HASH, "thresholds_db": list(VV_THRESHOLDS_DB),
    "analysis_grid_m": ANALYSIS_SCALE_M,
    "mask_restored_after_spatial_filter": True,
    "ee_sampled_polygon_area_km2": reference_area_m2 / 1e6,
    "total_months": int(len(monthly)),
    "months_with_scenes": int((monthly.scene_count > 0).sum()),
    "eligible_months": int(monthly.comparison_eligible.sum()),
    "single_scene_months": int(monthly.single_scene_warning.sum()),
    "annual_rasters_downloaded": int(sum((RASTERS / f"HAKALUKI_{year}_{CONFIG_HASH}_lowVV_frequency_30m.tif").exists() for year in range(START_YEAR, END_YEAR+1))),
    "validation": "not performed",
    "june2022_spatial_raster_exported": (RASTERS / f"HAKALUKI_JUNE2022_vs_2019_2025_{CONFIG_HASH}_30m.tif").exists(),
    "generated_utc": datetime.now(timezone.utc).isoformat(),
}
manifest_path = TABLES / "inundation09_run_manifest.json"
manifest_path.write_text(json.dumps(manifest, indent=2), encoding="utf-8")
print(json.dumps(manifest, indent=2))
print("\nKey results: ", TABLES / "inundation09_monthly_2019_2025.csv")
print("Annual figure:", FIGURES / "inundation09_annual_lowvv_frequency_2019_2025.png")
print("Public README should label these as unvalidated candidate low-VV screening products.")

In [ ]:

# ============================================================
# NOTEBOOK 09I — JUNE 2022 SPATIAL DEPARTURE SUMMARY
# Descriptive low-VV screening diagnostics only.
# ============================================================

from pathlib import Path
from shapely.geometry import mapping
from rasterio.features import geometry_mask
import rasterio
import numpy as np
import pandas as pd

ROOT = Path(
    r"F:\Wetland Test\Hakaluki_Haor_Wetland_Test_Jupyter_GEE_Project"
)

RASTERS = ROOT / "results/rasters/inundation_2019_2025"
TABLES = ROOT / "results/tables"
TABLES.mkdir(parents=True, exist_ok=True)

# Uses the exact processing configuration from Notebook 09.
raster_path = (
    RASTERS /
    f"HAKALUKI_JUNE2022_vs_2019_2025_{CONFIG_HASH}_30m.tif"
)

if not raster_path.is_file():
    raise FileNotFoundError(raster_path)

with rasterio.open(raster_path) as src:
    assert src.count == 4
    assert src.crs.to_epsg() == 32646

    data = (
        src.read(masked=True)
        .astype(np.float64)
        .filled(np.nan)
    )

    binary, baseline, departure, n_baseline = data

    pixel_area_km2 = abs(
        src.transform.a * src.transform.e
        - src.transform.b * src.transform.d
    ) / 1e6

    roi_utm = haor.to_crs(src.crs).geometry.union_all()

    inside = geometry_mask(
        [mapping(roi_utm)],
        out_shape=src.shape,
        transform=src.transform,
        invert=True
    )

valid = (
    inside
    & np.isfinite(binary)
    & np.isfinite(baseline)
    & np.isfinite(departure)
    & np.isfinite(n_baseline)
    & (n_baseline >= MIN_BASELINE_YEARS)
)

if not valid.any():
    raise RuntimeError("No valid spatial comparisons.")

if not np.allclose(
    departure[valid],
    binary[valid] - baseline[valid],
    atol=1e-5
):
    raise RuntimeError(
        "Departure raster is inconsistent with input bands."
    )

positive = valid & (departure > 1e-5)
negative = valid & (departure < -1e-5)
unchanged = valid & (np.abs(departure) <= 1e-5)

n_valid = int(valid.sum())

june_area = (
    np.sum(binary[valid]) * pixel_area_km2
)

baseline_mean_area = (
    np.sum(baseline[valid]) * pixel_area_km2
)

net_difference_area = (
    np.sum(departure[valid]) * pixel_area_km2
)

summary = pd.DataFrame([{
    "comparison": "June 2022 versus other Junes",
    "config_hash": CONFIG_HASH,
    "threshold_db": float(VV_THRESHOLDS_DB[1]),
    "valid_pixels": n_valid,
    "pixel_area_km2": pixel_area_km2,
    "area_compared_km2": n_valid * pixel_area_km2,
    "positive_departure_area_km2":
        int(positive.sum()) * pixel_area_km2,
    "negative_departure_area_km2":
        int(negative.sum()) * pixel_area_km2,
    "zero_departure_area_km2":
        int(unchanged.sum()) * pixel_area_km2,
    "positive_departure_percent":
        100 * positive.sum() / n_valid,
    "negative_departure_percent":
        100 * negative.sum() / n_valid,
    "june2022_lowvv_km2": june_area,
    "other_junes_mean_frequency_equivalent_km2":
        baseline_mean_area,
    "net_lowvv_difference_km2":
        net_difference_area,
    "mean_pixel_departure":
        float(np.mean(departure[valid])),
    "median_pixel_departure":
        float(np.median(departure[valid])),
    "baseline_years_min":
        int(np.min(n_baseline[valid])),
    "baseline_years_max":
        int(np.max(n_baseline[valid])),
    "validated_flood_area": False
}])

print("=== JUNE 2022 SPATIAL SCREENING SUMMARY ===")
display(summary.T)

print("\nBASELINE OBSERVATION SUPPORT")
counts, frequencies = np.unique(
    n_baseline[valid].astype(int),
    return_counts=True
)
display(pd.DataFrame({
    "baseline_years": counts,
    "pixels": frequencies
}))

out = TABLES / "inundation09_june2022_spatial_summary.csv"
summary.to_csv(out, index=False)

print("\nSaved:", out)
print("No new satellite processing performed.")
print("Results are not validated flood-area estimates.")
